# Validation C — the full cell

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/validation_c_pv.ipynb)

**Akerboom *et al.*, *ACS Photonics* 9 (2022) 3831–3840,
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389)**

## The physics

Optics, heat and electricity are coupled. The absorptance sets both how much
sunlight the cell converts and how hot it runs; the temperature sets the
open-circuit voltage; the operating voltage feeds back into the balance through
luminescent emission. radcoolpv solves the operating point and the temperature
together by fixed-point iteration.

The cell is a single diode with series and shunt resistance, an Auger term, and
a radiative saturation current from detailed balance. The band gap follows
Varshni, and its wavelength cuts off every photogeneration integral:

$$E_g(T) = E_{g0} - \frac{\alpha T^2}{T+\beta}, \qquad
J_\mathrm{sc} = q\!\int_0^{\lambda_g}\!\mathrm{IQE}\,\langle A_\mathrm{Si}\rangle\,\Phi_\mathrm{sun}\,d\lambda$$

Only $A_\mathrm{Si}$ makes carriers; the *full* absorptance heats the module.
Parasitic absorption is a thermal load and nothing more.

This group needs the **lossy** silicon table, not the nonabsorbing one groups A
and B use. Lossless silicon absorbs no sunlight at all, so $J_\mathrm{sc}$
integrates to zero and the cell reports a few millivolts — a failure that looks
like a solver bug and is a materials choice.

## Main result

| Surface | $T_\mathrm{eq}$ | Efficiency | MPP | $\beta_P$ |
|---|---:|---:|---:|---:|
| Bare Au/Si | 350.5 K | 14.17% | 142.6 W/m² | −0.303 %/K |
| Flat silica | 329.3 K | 18.09% | 182.1 W/m² | −0.299 %/K |
| Silica cylinders | 327.0 K | 18.64% | 187.7 W/m² | −0.300 %/K |

The temperature drops match the paper: 21.2 K bare → flat silica against 21 K,
2.3 K flat → cylinders against 3 K, 23.5 K bare → cylinders against 24 K.

Now read the efficiency column against the absorbed sunlight: 507.5 → 598.6 →
614.3 W/m². **Two mechanisms are at work, and the numbers separate them.**
The silica cools the module, and it also acts as an antireflection coating
that lets more light in. Both raise the efficiency; the exercise below pulls
them apart.

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## Build the solver

This group computes the optics from the geometry, so it needs S4 — a C++
extension with no PyPI package, built from source in about ten minutes.

In [ ]:
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"   # the tested revision

def build_s4():
    import importlib, importlib.util
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable."); return
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential", "git",
                    "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
                    "libopenblas-dev", "libsuitesparse-dev"], check=True)
    src = Path("/content/S4")
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/phoebe-p/S4.git", str(src)], check=True)
    subprocess.run(["git", "checkout", S4_COMMIT], cwd=src, check=True)
    subprocess.run(["make", "-j2", "S4_pyext"], cwd=src, check=True)
    importlib.invalidate_caches()

build_s4()

## The case

The full solar-to-thermal-infrared range. The upper limit is set by gold:
`RII_Olmon_2012_ev_Au` is tabulated to 24.93 µm and the loader refuses to
extrapolate rather than inventing values.

`n` and the angular grid are reduced below so this finishes. **The converged
settings — `n: 1000`, `hemisphere_theta_points: 8`, `s4_modes: 60` — take about
an hour per case**, longer than a free Colab runtime is guaranteed to last.
Treat what this cell prints as a smoke test.

In [ ]:
%%writefile validation_c.yaml
run:
  optics: true
  thermal: true
  plots: true
  mode: standard
  write_outputs: true
  results_dir: results/validation_c

simulation:
  wavelength: {min: 0.3, max: 24.9, n: 120}   # converged: 1000
  angles: hemispherical
  polarization: unpolarized
  hemisphere_theta_points: 3                  # converged: 8
  hemisphere_azimuth_points: 1
  s4_modes: 15                                # converged: 60

geometry:
  source: s4
  shape: cylinder
  photonic_material: sio2
  lattice: {type: hexagonal, x: 10.608811, y: 6.125}
  cylinder: {radius: 1.75, height: 2.25}

structure:
  - {material: sio2, thickness: 500.0}
  - {material: silicon, thickness: 500.0}
  - {material: gold, thickness: 0.08}
  - {material: vacuum, thickness: 0.0, terminal: true}

materials:
  sio2: PalikKitamura_SiO2
  silicon: Palik_Si            # lossy: a lossless cell makes no current
  gold: RII_Olmon_2012_ev_Au

thermal:
  ambient_temperature: 300.0
  convection_coefficient: 6.0
  voltage: {min: 0.1, max: 0.8, n: 60}
  equilibrium: auto

In [ ]:
CASE = "validation_c.yaml"
ctx = pipeline.run(config.load_cases(CASE)[0])
report.summary(ctx)

**Exercise.** Run it once with `photonic_material: vacuum`, `shape: flat` and the silicon/gold/vacuum stack. Compare `absorbed_solar_power` and `efficiency_equilibrium` with the cylinders. How much of the efficiency gain is cooling, and how much is simply letting more light in?

## Use your own material

This case computes the optics from the geometry, so what it needs from you is
an **optical constants table**, not a spectrum. Upload a CSV whose first line is
`lambda_um,n,k`: wavelength in micrometres, then the real and imaginary parts of
the refractive index.

The filename is the model name. `MyGlass.csv` becomes usable as `MyGlass` in the
`materials:` block above, so you can swap it in for `sio2` and re-run. Its
wavelength range has to cover the range the case asks for — the loader raises
rather than extrapolating past the last tabulated point.


In [ ]:
from google.colab import files
from radcoolpv.materials import registry

MATERIALS = PROJECT / "radcoolpv" / "materials" / "data"

for name, blob in files.upload().items():
    header = blob[:200].decode(errors="ignore").splitlines()[0].strip().lower()
    if name.lower().endswith(".csv") and header.startswith("lambda_um,"):
        (MATERIALS / name).write_bytes(blob)
        print(f"{name}: installed as material {Path(name).stem!r}")
    else:
        print(f"{name}: not installed. Expected a .csv whose first line is "
              f"'lambda_um,n,k'; this one starts {header[:40]!r}")

print("\navailable materials:", ", ".join(sorted(registry.available())))
